# 05 — Equity-Weighted MCLP

Reweight demand upward for census tracts with **low median household income** and
**high zero-vehicle household rates**, incentivising the optimiser to cover
underserved areas without replacing the original NKDE demand signal.

| File consumed | Purpose |
|---|---|
| `data/processed/candidate_sites_filtered.gpkg` | 7,644 candidates (EPSG:32618) |
| `data/processed/coverage_dict.pkl` | network coverage dict from Stage 4 |
| `data/processed/mclp_selected_p{p}.gpkg` | original MCLP baselines |

**Outputs** → `data/processed/mclp_equity_p{p}.gpkg`, `equity_comparison_p100.png`, `equity_nta_comparison.png`

In [ ]:
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import requests
import pulp

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

TARGET_CRS = 'EPSG:32618'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed'
IMG_DIR = PROJECT_ROOT / 'data' / 'outputs'
IMG_DIR.mkdir(parents=True, exist_ok=True)

candidates = gpd.read_file(PROJECT_ROOT / 'data/processed/candidate_sites_filtered.gpkg')
candidates = candidates.to_crs(TARGET_CRS)
print(f'Candidates: {len(candidates):,}  CRS: {candidates.crs}')

with open(PROJECT_ROOT / 'data/processed/coverage_dict.pkl', 'rb') as f:
    coverage = pickle.load(f)
print(f'Coverage dict loaded: {len(coverage):,} entries')

## Step 1 — ACS data for Manhattan census tracts

Pull ACS 5-year 2022 at tract level from the Census Bureau API (no key required
for low-volume use):

- **B19013_001E** — median household income
- **B08201_001E** — total households
- **B08201_002E** — zero-vehicle households

Tract polygons come from Census TIGER/Line 2020. Result cached to `manhattan_acs.gpkg`.

In [ ]:
import os

ACS_PATH = PROJECT_ROOT / 'data/processed/manhattan_acs.gpkg'

if ACS_PATH.exists():
    tracts = gpd.read_file(ACS_PATH)
    print(f'Loaded cached ACS data: {len(tracts)} tracts')
else:
    # ACS 5-year 2006-2010 requires a Census API key.
    # Get a free key at https://api.census.gov/data/key_signup.html
    # then set:  set CENSUS_API_KEY=<your_key>  (Windows) or export CENSUS_API_KEY=<key>
    api_key = os.environ.get('CENSUS_API_KEY', '')
    if not api_key:
        raise EnvironmentError(
            'CENSUS_API_KEY environment variable is not set.\n'
            'The 2010 ACS vintage requires a free Census API key.\n'
            'Sign up at: https://api.census.gov/data/key_signup.html\n'
            'Then run:   set CENSUS_API_KEY=<your_key>   before launching Jupyter.'
        )

    census_url = 'https://api.census.gov/data/2010/acs/acs5'
    params = {
        'get': 'NAME,B19013_001E,B08201_001E,B08201_002E',
        'for': 'tract:*',
        'in': 'state:36 county:061',
        'key': api_key,
    }
    print('Fetching ACS 2006-2010 data from Census API...')
    resp = requests.get(census_url, params=params, timeout=30)
    if resp.status_code != 200 or resp.text.strip().startswith('<'):
        raise RuntimeError(
            f'Census API returned unexpected response (HTTP {resp.status_code}).\n'
            f'First 300 chars: {resp.text[:300]}'
        )
    raw = resp.json()
    acs_df = pd.DataFrame(raw[1:], columns=raw[0])

    for col in ['B19013_001E', 'B08201_001E', 'B08201_002E']:
        acs_df[col] = pd.to_numeric(acs_df[col], errors='coerce')
        acs_df.loc[acs_df[col] < 0, col] = np.nan  # Census encodes suppressed as -666666666

    print(f'ACS rows fetched: {len(acs_df)}')

    # --- TIGER tract boundaries (New York State, then filter to county 061) ---
    tiger_url = 'https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_36_tract.zip'
    print('Downloading Census TIGER tract boundaries for New York State...')
    tracts_raw = gpd.read_file(tiger_url)
    tracts_mn = tracts_raw[tracts_raw['COUNTYFP'] == '061'].copy()
    print(f'Manhattan TIGER tracts: {len(tracts_mn)}')

    # --- Join attributes to geometries ---
    tracts_mn = tracts_mn.merge(
        acs_df,
        left_on='TRACTCE',
        right_on='tract',
        how='left',
    )
    tracts_mn = tracts_mn.rename(columns={
        'B19013_001E': 'med_income',
        'B08201_001E': 'total_hh',
        'B08201_002E': 'zero_veh_hh',
    })
    tracts = tracts_mn[['geometry', 'GEOID', 'TRACTCE', 'med_income', 'total_hh', 'zero_veh_hh']].copy()
    tracts.to_file(ACS_PATH, driver='GPKG')
    print(f'Saved -> {ACS_PATH.name}')

print(tracts[['TRACTCE', 'med_income', 'total_hh', 'zero_veh_hh']].head())
n_miss = tracts['med_income'].isna().sum()
print(f'Missing med_income: {n_miss} tracts')
inc_min = tracts['med_income'].min()
inc_max = tracts['med_income'].max()
print(f'Income range: ${inc_min:,.0f} - ${inc_max:,.0f}')

## Step 2 — Compute equity weights

For each census tract:

```
income_score   = 1 - (tract_income / max_income)   # lower income → higher score
zero_car_rate  = zero_vehicle_households / total_households
equity_weight  = 0.5 * income_score + 0.5 * zero_car_rate
```

Then normalise `equity_weight` to [0, 1] across all tracts.

In [ ]:
tracts = tracts.copy()

# income_score: lower income -> higher score; clamp to [0,1]
max_income = tracts['med_income'].max()
tracts['income_score'] = (1 - tracts['med_income'] / max_income).clip(0, 1)
# Fill tracts with suppressed income data with the median score (conservative imputation)
median_inc_score = tracts['income_score'].median()
tracts['income_score'] = tracts['income_score'].fillna(median_inc_score)

# zero_car_rate: fraction of households with no vehicle
tracts['zero_car_rate'] = (tracts['zero_veh_hh'] / tracts['total_hh']).clip(0, 1).fillna(0)

# equity_weight: equal blend, then normalise to [0, 1]
tracts['equity_weight_raw'] = 0.5 * tracts['income_score'] + 0.5 * tracts['zero_car_rate']
w_min = tracts['equity_weight_raw'].min()
w_max = tracts['equity_weight_raw'].max()
tracts['equity_weight'] = (tracts['equity_weight_raw'] - w_min) / (w_max - w_min)

print('Equity weight summary:')
print(tracts['equity_weight'].describe().round(3))
print()
print('Top 5 highest-weight tracts:')
cols = ['TRACTCE', 'med_income', 'zero_car_rate', 'equity_weight']
print(tracts.nlargest(5, 'equity_weight')[cols].to_string(index=False))

## Step 3 — Join equity weights to candidates

Spatially join each candidate to its census tract, then compute:

```
equity_adjusted_score = nkde_score * (1 + equity_weight)
```

This boosts demand in low-income, low-vehicle tracts without replacing the
original NKDE signal.

In [ ]:
tracts_proj = tracts.to_crs(TARGET_CRS)

# Reset to 0-based index so coverage dict keys (also 0-based) stay aligned
cands = candidates.copy().reset_index(drop=True)

cands_joined = gpd.sjoin(
    cands,
    tracts_proj[['geometry', 'equity_weight']],
    how='left',
    predicate='within',
)
# Keep first match for the rare boundary cases (duplicate rows with same index)
cands_joined = cands_joined[~cands_joined.index.duplicated(keep='first')]
cands_joined = cands_joined.drop(columns=['index_right'], errors='ignore')

# Candidates outside any tract (e.g., waterfront edge) get zero equity boost
n_unmatched = cands_joined['equity_weight'].isna().sum()
cands_joined['equity_weight'] = cands_joined['equity_weight'].fillna(0.0)

cands_joined['equity_adjusted_score'] = (
    cands_joined['nkde_score'] * (1 + cands_joined['equity_weight'])
)

print(f'Candidates after join: {len(cands_joined):,}')
print(f'Unmatched (equity_weight=0): {n_unmatched:,}')
print()
print('Score comparison:')
print(cands_joined[['nkde_score', 'equity_adjusted_score']].describe().round(4))
boost = (cands_joined['equity_adjusted_score'] / cands_joined['nkde_score'].replace(0, np.nan)).mean()
print(f'Mean multiplicative boost: {boost:.3f}x')

## Step 4 — Equity-weighted MCLP

Same MCLP formulation as Stage 4 but the objective uses `equity_adjusted_score`
as demand weights. The coverage dict is reused unchanged.

Run for p ∈ {50, 75, 100, 125, 150}.

In [ ]:
import gc

def run_equity_mclp(candidates_gdf, coverage_matrix, p, score_col='equity_adjusted_score'):
    t0 = time.time()
    n = len(candidates_gdf)
    scores = candidates_gdf[score_col].values

    prob = pulp.LpProblem(f'MCLP_equity_p{p}', pulp.LpMaximize)
    x = pulp.LpVariable.dicts('x', range(n), cat='Binary')
    y = pulp.LpVariable.dicts('y', range(n), lowBound=0, upBound=1, cat='Continuous')

    prob += pulp.lpSum(scores[i] * y[i] for i in range(n))

    for i, js in coverage_matrix.items():
        prob += pulp.lpSum(x[j] for j in js) >= y[i]

    prob += pulp.lpSum(x[j] for j in range(n)) == p

    prob.solve(pulp.PULP_CBC_CMD(msg=1, gapRel=0.005, timeLimit=300))

    sel_idx = [j for j in range(n) if (pulp.value(x[j]) or 0) > 0.5]
    cov_dem = sum(scores[i] for i in range(n) if (pulp.value(y[i]) or 0) > 0.5)
    pct = cov_dem / scores.sum() * 100

    sel_gdf = candidates_gdf.iloc[sel_idx].copy().reset_index(drop=True)
    del prob, x, y
    gc.collect()
    return sel_gdf, cov_dem, pct, time.time() - t0


p_values = [50, 75, 100, 125, 150]
equity_results = {}

_all_saved = all((OUT_DIR / f'mclp_equity_p{p}.gpkg').exists() for p in p_values)
if _all_saved:
    print('Loading saved equity results from disk (skipping re-optimisation)...')
    for p in p_values:
        sel_gdf = gpd.read_file(OUT_DIR / f'mclp_equity_p{p}.gpkg').to_crs(TARGET_CRS)
        equity_results[p] = {'selected_gdf': sel_gdf}
        print(f'  p={p:3d}: loaded {len(sel_gdf)} sites from mclp_equity_p{p}.gpkg')
else:
    for p in p_values:
        sel_gdf, cov_dem, pct, runtime = run_equity_mclp(cands_joined, coverage, p)
        equity_results[p] = {'selected_gdf': sel_gdf, 'covered_demand': cov_dem, 'pct_covered': pct}
        sel_gdf.to_file(OUT_DIR / f'mclp_equity_p{p}.gpkg', driver='GPKG')
        print(
            f'p={p:3d}: {len(sel_gdf)} sites | '
            f'equity demand covered = {cov_dem:.2f} ({pct:.1f}%) | '
            f'runtime = {runtime:.1f}s'
        )
        gc.collect()

## Step 5 — Comparison analysis

For a fair apples-to-apples comparison, both selections are evaluated on the
original **nkde_score** demand. Equity MCLP may sacrifice a few percentage points
of raw efficiency in exchange for better coverage of underserved areas.

In [ ]:
nkde_vals = cands_joined['nkde_score'].values


def nkde_pct_covered(sel_idx_set, coverage_dict, demand):
    """% of total nkde demand covered by the given set of site indices."""
    covered = sum(
        demand[i]
        for i, js in coverage_dict.items()
        if sel_idx_set.intersection(js)
    )
    return covered / demand.sum() * 100


def match_to_candidates(sites_gdf, cands_gdf):
    """Map loaded site geometries back to candidate indices via nearest match (<= 10 m)."""
    sites_proj = sites_gdf.to_crs(TARGET_CRS)[['geometry']].reset_index(drop=True)
    match = gpd.sjoin_nearest(
        sites_proj,
        cands_gdf[['geometry']].reset_index(),  # preserves 0-based integer index
        how='left',
        max_distance=10.0,
    )
    return set(match['index'].dropna().astype(int))


header = f'{"p":>4}  {"Orig nkde%":>10}  {"Equity nkde%":>13}  {"Delta%":>7}  {"Into equity":>11}  {"Out of equity":>13}'
print(header)
print('-' * len(header))

for p in p_values:
    orig_gdf = gpd.read_file(PROJECT_ROOT / f'data/processed/mclp_selected_p{p}.gpkg')
    orig_idx = match_to_candidates(orig_gdf, cands_joined)
    eq_idx = match_to_candidates(equity_results[p]['selected_gdf'], cands_joined)

    orig_pct = nkde_pct_covered(orig_idx, coverage, nkde_vals)
    eq_pct = nkde_pct_covered(eq_idx, coverage, nkde_vals)
    delta = eq_pct - orig_pct
    shifted_in = len(eq_idx - orig_idx)
    shifted_out = len(orig_idx - eq_idx)

    print(
        f'{p:>4}  {orig_pct:>9.1f}%  {eq_pct:>12.1f}%  '
        f'{delta:>+7.1f}  {shifted_in:>11}  {shifted_out:>13}'
    )

## Step 6 — Map comparison at p = 50

Side-by-side: original MCLP (blue) vs equity MCLP (red) for p = 50.

In [ ]:
import gc
# Free large data structures no longer needed for plotting
del coverage
gc.collect()

orig_p50 = gpd.read_file(PROJECT_ROOT / 'data/processed/mclp_selected_p50.gpkg').to_crs(TARGET_CRS)
eq_p50 = equity_results[50]['selected_gdf'].copy()

# Common extent in EPSG:3857 (required by contextily)
orig_wm = orig_p50.to_crs('EPSG:3857')
eq_wm = eq_p50.to_crs('EPSG:3857')
all_x = list(orig_wm.geometry.x) + list(eq_wm.geometry.x)
all_y = list(orig_wm.geometry.y) + list(eq_wm.geometry.y)
pad = 1500
xlim = (min(all_x) - pad, max(all_x) + pad)
ylim = (min(all_y) - pad, max(all_y) + pad)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

for ax, gdf_wm, title, color in [
    (axes[0], orig_wm, 'Original MCLP  p=50', '#1f78b4'),
    (axes[1], eq_wm,   'Equity MCLP  p=50',   '#e31a1c'),
]:
    ax.scatter(gdf_wm.geometry.x, gdf_wm.geometry.y, c=color, s=80, zorder=5, edgecolors='none')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    try:
        import contextily as ctx
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    except Exception:
        ax.set_facecolor('#f0f0f0')
    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

plt.suptitle('MCLP Site Selection: Original vs. Equity-Weighted  (p = 50)', fontsize=14, y=1.01)
plt.subplots_adjust(wspace=0.05)
out_path = IMG_DIR / 'equity_comparison_p50.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved -> {out_path.name}')

## Step 7 — NTA-level equity analysis

For each Manhattan Neighbourhood Tabulation Area (NTA), compute the share of
its area covered by a 500 m service buffer around the p = 100 selected sites,
then plot the difference (equity minus original) sorted descending.

In [ ]:
NTA_PATH = PROJECT_ROOT / 'data/raw/nta_boundaries.gpkg'

# Invalidate cache if it has no attribute columns (from a bad prior download)
if NTA_PATH.exists():
    _tmp = gpd.read_file(NTA_PATH)
    if len([c for c in _tmp.columns if c != 'geometry']) == 0:
        print('Cached NTA file has no attribute columns — deleting and re-downloading...')
        NTA_PATH.unlink()

if not NTA_PATH.exists():
    import io
    nta_url = 'https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson?$limit=500'
    print('Downloading NYC 2020 NTA boundaries from NYC Open Data...')
    resp = requests.get(nta_url, timeout=60)
    resp.raise_for_status()
    nta_raw = gpd.read_file(io.BytesIO(resp.content))
    print(f'Downloaded columns: {list(nta_raw.columns)}')
    nta_raw.to_file(NTA_PATH, driver='GPKG')
    print(f'Saved -> {NTA_PATH.name}')

nta = gpd.read_file(NTA_PATH).to_crs(TARGET_CRS)
print(f'NTA columns: {list(nta.columns)}')

# Detect column names (field names differ between 2010 and 2020 Open Data releases)
attr_cols = [c for c in nta.columns if c != 'geometry']
name_col = next(
    (c for c in attr_cols if c.lower() in ('ntaname', 'nta_name', 'ntanam', 'nta2020')),
    attr_cols[0] if attr_cols else None,
)
boro_col = next(
    (c for c in attr_cols if 'boro' in c.lower() and 'name' in c.lower()),
    None,
)
print(f'Using name_col={name_col!r}, boro_col={boro_col!r}')

if boro_col:
    nta_mn = nta[nta[boro_col] == 'Manhattan'].copy()
else:
    nta_mn = nta.copy()
print(f'Manhattan NTAs: {len(nta_mn)}')


def nta_coverage_pct(sites_gdf, nta_gdf, buffer_m=500):
    """For each NTA polygon, return % of area within buffer_m of any selected site."""
    buf = sites_gdf.to_crs(TARGET_CRS).buffer(buffer_m).unary_union
    result = nta_gdf.copy()
    result['nta_area'] = result.geometry.area
    result['covered_area'] = result.geometry.intersection(buf).area
    result['pct_covered'] = (result['covered_area'] / result['nta_area'] * 100).clip(0, 100)
    return result


for p in [50, 100]:
    orig_gdf = gpd.read_file(PROJECT_ROOT / f'data/processed/mclp_selected_p{p}.gpkg').to_crs(TARGET_CRS)
    eq_gdf = equity_results[p]['selected_gdf'].copy()

    nta_orig = nta_coverage_pct(orig_gdf, nta_mn)
    nta_eq = nta_coverage_pct(eq_gdf, nta_mn)

    nta_cmp = (
        nta_orig[[name_col, 'pct_covered']]
        .rename(columns={'pct_covered': 'orig_pct'})
        .merge(
            nta_eq[[name_col, 'pct_covered']].rename(columns={'pct_covered': 'eq_pct'}),
            on=name_col,
        )
    )
    nta_cmp['diff'] = nta_cmp['eq_pct'] - nta_cmp['orig_pct']
    nta_cmp = nta_cmp.sort_values('diff', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(16, 7))
    colors = [
        '#d73027' if d > 0.5 else '#4575b4' if d < -0.5 else '#999999'
        for d in nta_cmp['diff']
    ]
    ax.bar(range(len(nta_cmp)), nta_cmp['diff'], color=colors, edgecolor='white', linewidth=0.3)
    ax.set_xticks(range(len(nta_cmp)))
    ax.set_xticklabels(nta_cmp[name_col], rotation=90, fontsize=7)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_ylabel('Coverage change (equity - original)  percentage points', fontsize=11)
    ax.set_title(
        f'NTA-level Coverage Change: Equity MCLP vs. Original MCLP  (p={p}, 500 m service radius)',
        fontsize=12,
    )
    legend_els = [
        Patch(facecolor='#d73027', label='Equity gains > 0.5 pp'),
        Patch(facecolor='#4575b4', label='Equity loses > 0.5 pp'),
        Patch(facecolor='#999999', label='Negligible change'),
    ]
    ax.legend(handles=legend_els, fontsize=9)
    plt.subplots_adjust(bottom=0.35)
    out_path = IMG_DIR / f'equity_nta_comparison_p{p}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {out_path.name}')
    print()
    print(f'Top 10 NTAs with greatest equity coverage gain (p={p}):')
    print(nta_cmp[[name_col, 'orig_pct', 'eq_pct', 'diff']].head(10).to_string(index=False))
    print()